# Spike: comparar Camino A vs Camino B

Propósito: decidir qué camino usar para QAOA según el plan (M1). Se usan los primeros 4x4 de `qap10` para probar N=3 y N=4.

**Fuentes (código que se ejecuta):**
- `src/spike/spike_common.py` — modelo compartido (F4/D4, `build_qp`, `decode`, `cost`).
- `src/spike/spike_qaoa_camino_a.py` — vía automática `MinimumEigenOptimizer(QAOA(...))` con `SamplerV2` de Aer. Incluye `QAOAQuasi` para corregir la incompatibilidad entre qiskit-algorithms 0.4 y qiskit-optimization 0.7.
- `src/spike/spike_qaoa_camino_b.py` — vía artesanal: `QAOAAnsatz` + `EstimatorV2` de Aer + `scipy.optimize.minimize(method="COBYLA")` + muestreo final con `SamplerV2` de Aer.

**Aviso importante:** Nunca usar `StatevectorSampler`/`StatevectorEstimator` de referencia (Python puro). Con N=3 tardan 87 s; con N=4 no son viables. Usar siempre las primitivas de `qiskit_aer.primitives`.

**Criterio:** si N=3/4 se resuelve en ~2-5 s con calidad aceptable, usarlo. El resultado del spike: **Camino B** cumple el tiempo; **Camino A** es lento con N=4 y necesitó parches.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))
sys.path.insert(0, str(Path('..') / 'src' / 'spike'))

from spike_qaoa_camino_a import run as run_a
from spike_qaoa_camino_b import run as run_b
from spike_common import print_report

## Camino A: caja negra

In [ ]:
for r in [run_a(n=3, reps=1, maxiter=100, shots=4096, seed=0, penalty=None),
          run_a(n=4, reps=1, maxiter=100, shots=4096, seed=0, penalty=None)]:
    print_report(r, limit_s=5)

## Camino B: artesanal (recomendado)

In [ ]:
for r in [run_b(n=3, reps=1, maxiter=100, shots=4096, seed=0, penalty=None),
          run_b(n=4, reps=2, maxiter=100, shots=4096, seed=0, penalty=None)]:
    print_report(r, limit_s=5)

## tests/test_model.py (lo que comprueba)

- `test_qubo_energy_equals_cost_on_permutations` (N=3,4): para cada permutación, la energía QUBO penalizada == `cost(perm)`. Las restricciones se satisfacen en los puntos factibles.
- `test_qp_reaches_control_optimum` (N=3,4): con `λ >= 1.5*λ*` el mínimo del QUBO es factible y óptimo (24/32). Se busca por fuerza bruta sobre todas las asignaciones binarias.
- `test_ising_offset_matches_ground_energy` (N=3,4): se comprueba `⟨x|H|x⟩ + offset == energía QUBO` para asignaciones factibles (relación `x = (1-Z)/2`).
- `test_decode_little_endian`: el bitstring medido es little-endian (el carácter más a la derecha = qubit 0). Hay que invertirlo antes de mapear a `x_{i,j}`.
- `test_penalty_threshold_positive`: `λ* > 0`.

In [ ]:
!cd .. && $env:PYTHONPATH='src' && uvx --with numpy --with pytest --with qiskit --with qiskit-aer --with qiskit-algorithms --with qiskit-optimization --with scipy pytest tests/test_model.py -v